## Задача 1. My heart will go on

Датасет **titanic** из библиотеки `Seaborn` содержит информацию о пассажирах легендарного корабля Титаник, который затонул в 1912 году после столкновения с айсбергом. Этот набор данных часто используется для обучения и тестирования алгоритмов машинного обучения, особенно в задачах бинарной классификации (выжил / не выжил).

**Описание данных**

| Поле         | Тип      | Описание |
|--------------|----------|----------|
| `survived`   | int      | Выжил (1) или не выжил (0) |
| `pclass`     | int      | Класс билета (1, 2, 3) |
| `sex`        | str      | Пол (`male`/`female`) |
| `age`        | float    | Возраст |
| `sibsp`      | int      | Количество братьев/сестёр/супругов на борту |
| `parch`      | int      | Количество родителей/детей на борту |
| `fare`       | float    | Стоимость билета |
| `embarked`   | str      | Порт посадки (`C`=Cherbourg, `Q`=Queenstown, `S`=Southampton) |
| `class`      | str      | Класс билета (`First`, `Second`, `Third`) |
| `who`        | str      | Категория: `man`, `woman` или `child` |
| `adult_male` | bool     | Является ли взрослым мужчиной |
| `deck`       | str      | Палуба |
| `embark_town`| str      | Название порта посадки |
| `alive`      | str      | Выжил (`yes`/`no`) |
| `alone`      | bool     | Путешествовал один |

**Загрузка датасета**

In [59]:
import seaborn as sns
import pandas as pd

titanic_data = sns.load_dataset("titanic")
titanic_data.sample(5)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
623,0,3,male,21.0,0,0,7.8542,S,Third,man,True,NaN,Southampton,no,True
682,0,3,male,20.0,0,0,9.2250,S,Third,man,True,NaN,Southampton,no,True
286,1,3,male,30.0,0,0,9.5000,S,Third,man,True,NaN,Southampton,yes,True
268,1,1,female,58.0,0,1,153.4625,S,First,woman,False,C,Southampton,yes,False
390,1,1,male,36.0,1,2,120.0000,S,First,man,True,B,Southampton,yes,False


**Задача**

Ниже описаны 10 небольших заданий, которые вам необходимо решить.

**Подсказка**:

В некоторых заданиях вам может быть полезен метод `value_counts`.

### Часть 1

Определите число пропущенных данных для каждого столбца таблицы `titanic_data`.

In [60]:
is_na = titanic_data.isna()
print(is_na.sum())

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64


### Часть 2

Удалите все столбцы, количество пропусков в которых превышает половину количества строк в таблице.

После того, как вы удалите все столбцы, нарушающие описанное условие, удалите все строки, количество пропусков в которых превышает половину количества столбцов.

In [61]:

thresh = len(titanic_data.index) * 0.5
titanic_data = titanic_data.dropna(thresh=thresh, axis=1)
thresh = len(titanic_data.columns) * 0.5
titanic_data = titanic_data.dropna(thresh=thresh, axis=0)


### Часть 3

Если вы сделали все правильно, больше всего пропусков должно остаться в столбце `"age"` - 177. Их необходимо заполнить. Заполним пропуски следующим образом:
- Если значение столбца `"who"="man"`, пропуск необходимо заполнить медианным значением известных возрастов мужчин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="woman"`, пропуск необходимо заполнить медианным значением известных возрастов женщин, округленным до ближайшего целого числа;
- Если значение столбца `"who"="child"`, пропуск необходимо заполнить медианным значением известных возрастов детей, округленным до ближайшего целого числа;

In [62]:
#is_na = titanic_data.isna()
#print(is_na.sum())

median_man = round(titanic_data[titanic_data["who"] == "man"].age.median())
median_woman = round(titanic_data[titanic_data["who"] == "woman"].age.median())
child_median = round(titanic_data[titanic_data["who"] == "child"].age.median())

mask_man = (titanic_data["who"] == "man") & (titanic_data["age"].isna())
mask_woman = (titanic_data["who"] == "woman") & (titanic_data["age"].isna())
mask_child = (titanic_data["who"] == "child") & (titanic_data["age"].isna())

titanic_data.loc[mask_man, "age"] = median_man
titanic_data.loc[mask_woman, "age"] = median_woman
titanic_data.loc[mask_child, "age"] = child_median


### Часть 4

Удалите все строки, в которых осталось больше одного пропуска. Если вы все сделали правильно, после этого действия в таблице не должно остаться пропусков.

In [63]:
thresh = len(titanic_data.columns) - 1
titanic_data = titanic_data.dropna(thresh=thresh, axis=0)



### Часть 5

Определите название города, из которого отправилось больше всего пассажиров.

In [64]:
cities = titanic_data["embark_town"].value_counts()
biggest = cities.index[0]
print(biggest)

Southampton


### Часть 6

Определите процент выживших пассажиров от числа пассажиров, оставшихся в таблице после очистки данных. Ответ округлите до 2 знаков после запятой.

In [65]:
survival = titanic_data["survived"].mean() * 100
print(f"{survival:.2f}%")

38.25%


### Часть 7

Определите число выживших пассажиров для каждого пункта отправления. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия пунктов отправления, а значения - число выживших пассажиров.

In [66]:
surv_emb = titanic_data[titanic_data["survived"] == 1]["embark_town"].value_counts()
surv_emb

embark_town
Southampton    217
Cherbourg       93
Queenstown      30
Name: count, dtype: int64

### Часть 8

Определите процент выживших пассажиров в каждом классе. Значения округлите до 2 знаков после запятой. В ответе должен получиться объект типа `pd.Series`, индексы которого - названия классов, а значения - процент выживших пассажиров.

In [67]:
res = {}

for titanic_class in titanic_data["class"].unique():
    mask_class = titanic_data["class"] == titanic_class
    surv_proc = titanic_data[mask_class]["survived"].mean() * 100
    res[titanic_class] = round(surv_proc, 2)
    
ans = pd.Series(res)
ans
    

    

Third     24.24
First     62.62
Second    47.28
dtype: float64

### Часть 9

Будем считать, что пассажиры, купившие билет **НЕ МЕНЕЕ** чем за $100, считаются богатыми. Определите процент выживших среди богатых пассажиров. Ответ округлите до 2 знаков после запятой. В ответе должно получиться число. 

In [68]:
wealth_mask = titanic_data["fare"] >=100
wealth_surv = titanic_data[wealth_mask]["survived"].mean() * 100
print(round(wealth_surv, 2))

73.58


### Часть 10

Определите количество детей, путешествовавших в одиночку.

In [72]:
child_alone_mask = (titanic_data["who"] == "child") & (titanic_data["alone"] == True)
child_alone = titanic_data[child_alone_mask]
print(len(child_alone.index))

6


Какие выводы вы можете сделать о выживших пассажирах Титаника? 